# Perkolace: kdy to proteče

*Nad rámec skripta — aplikace.*

---

Mřížka trubek, z nichž je každá nezávisle průchodná s pravděpodobností $p$. Proteče voda zleva doprava?

Pod jistou hodnotou $p$ prakticky nikdy, nad ní prakticky vždy — a přechod mezi tím je tím ostřejší, čím větší mřížku vezmeme. V nekonečné mřížce je **úplně ostrý**: existuje kritická hodnota $p_c$, pod kterou je průsak nemožný a nad kterou jistý.

Tomu se říká **fázový přechod** a je to táž věc jako ve `vetvici_proces`, jen v prostoru místo v čase. Stejná matematika popisuje šíření požáru v lese, poruchu v rozvodné síti, nebo průsak ropy pískovcem.

Model má vzácnou vlastnost: pro vazbovou perkolaci na čtvercové mřížce je

$$p_c = \tfrac12 \qquad \textbf{přesně},$$

což plyne ze samoduality mřížky (Kesten, 1980). Máme tedy opět přesnou kontrolu — a navíc nejhezčí obrázky v repozitáři.

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def mrizka(radku, sloupcu, p):
    """Náhodné otevření vazeb. H[r][c] spojuje (r,c) a (r,c+1), V[r][c] spojuje (r,c) a (r+1,c)."""
    H = [[random.random() < p for _ in range(sloupcu - 1)] for _ in range(radku)]
    V = [[random.random() < p for _ in range(sloupcu)] for _ in range(radku - 1)]
    return H, V


def dosazitelne(radku, sloupcu, H, V):
    """Které uzly jsou dosažitelné z levého okraje (prohledávání do hloubky)."""
    videno = [[False] * sloupcu for _ in range(radku)]
    zasobnik = []
    for r in range(radku):
        videno[r][0] = True
        zasobnik.append((r, 0))
    while zasobnik:
        r, c = zasobnik.pop()
        for rr, cc, otevreno in ((r, c + 1, c < sloupcu - 1 and H[r][c]),
                                 (r, c - 1, c > 0 and H[r][c - 1]),
                                 (r + 1, c, r < radku - 1 and V[r][c]),
                                 (r - 1, c, r > 0 and V[r - 1][c])):
            if otevreno and not videno[rr][cc]:
                videno[rr][cc] = True
                zasobnik.append((rr, cc))
    return videno


def protece(radku, sloupcu, p):
    H, V = mrizka(radku, sloupcu, p)
    videno = dosazitelne(radku, sloupcu, H, V)
    return any(videno[r][sloupcu - 1] for r in range(radku))

## 1. Jak to vypadá

Nejdřív obrázek. Pro tři hodnoty $p$ nakreslíme otevřené vazby a zvýrazníme, kam se voda z levého okraje dostane.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
L = 40
for ax, p in zip(axes, (0.40, 0.50, 0.60)):
    H, V = mrizka(L, L, p)
    videno = dosazitelne(L, L, H, V)
    for r in range(L):
        for c in range(L - 1):
            if H[r][c]:
                mokro = videno[r][c] and videno[r][c + 1]
                ax.plot([c, c + 1], [r, r], color="tab:blue" if mokro else "0.82",
                        lw=1.1 if mokro else 0.6, zorder=2 if mokro else 1)
    for r in range(L - 1):
        for c in range(L):
            if V[r][c]:
                mokro = videno[r][c] and videno[r + 1][c]
                ax.plot([c, c], [r, r + 1], color="tab:blue" if mokro else "0.82",
                        lw=1.1 if mokro else 0.6, zorder=2 if mokro else 1)
    prosaklo = any(videno[r][L - 1] for r in range(L))
    ax.set_title(f"$p$ = {p:.2f} — {'proteklo' if prosaklo else 'neproteklo'}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

Vlevo je modrá oblast malá a roztříštěná do ostrůvků. Vpravo tvoří jeden velký propojený celek, který projde skrz. Uprostřed, přesně v kritickém bodě, je to na hraně — a právě tam má obrazec tu charakteristickou „děravou" strukturu na všech velikostech zároveň.

## 2. Přechod se zostřuje s velikostí

Teď to změříme. Pro několik velikostí $L$ projdeme $p$ a spočítáme podíl mřížek, kterými proteče.

In [ ]:
def krivka(L, pecka, M_=400):
    return [sum(protece(L, L, p) for _ in range(M_)) / M_ for p in pecka]


pecka = [0.30 + 0.02 * i for i in range(21)]
velikosti = (8, 16, 32, 64)
krivky = {L: krivka(L, pecka, 400 if L <= 32 else 200) for L in velikosti}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
ax = axes[0]
for L in velikosti:
    ax.plot(pecka, krivky[L], "o-", ms=3, label=f"$L$ = {L}")
ax.axvline(0.5, color="k", ls=":", lw=1)
ax.axhline(0.5, color="k", ls=":", lw=1)
ax.text(0.505, 0.05, "$p_c = 1/2$", fontsize=9)
ax.set_xlabel("pravděpodobnost otevření vazby $p$")
ax.set_ylabel("P(proteče)")
ax.set_title("Čím větší mřížka, tím ostřejší zlom", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
print(f"{'L':>5s} {'čtverec L x L':>20s} {'obdélník L x (L+1)':>24s}")
ctverec, obdelnik = [], []
for L in velikosti:
    M_ = 12_000 if L <= 32 else 4_000
    fc = sum(protece(L, L, 0.5) for _ in range(M_)) / M_
    fo = sum(protece(L, L + 1, 0.5) for _ in range(M_)) / M_
    se = math.sqrt(0.25 / M_)
    ctverec.append((fc, se)); obdelnik.append((fo, se))
    print(f"{L:>5d} {fc:>12.4f} +- {se:.4f} {fo:>16.4f} +- {se:.4f}")

ax.errorbar(velikosti, [f for f, _ in ctverec], yerr=[s for _, s in ctverec],
            fmt="o-", capsize=4, label="čtverec $L \\times L$")
ax.errorbar(velikosti, [f for f, _ in obdelnik], yerr=[s for _, s in obdelnik],
            fmt="s-", capsize=4, color="tab:green", label="obdélník $L \\times (L+1)$")
ax.axhline(0.5, color="r", ls="--", label="přesně 1/2")
ax.set_xscale("log", base=2)
ax.set_xlabel("velikost mřížky $L$ (log)")
ax.set_ylabel("P(proteče) při $p = 1/2$")
ax.set_ylim(0.44, 0.60)
ax.set_title("Přesně 1/2 platí jen pro ten správný obdélník", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nNa čtverci vychází P(proteče) nad 1/2 a klesá k ní až s rostoucím L —")
print("čtverec totiž zvýhodňuje vodorovný směr. Samodualita dává PŘESNĚ 1/2")
print("pro obdélník o jeden sloupec širší, a tam to sedí pro každé L zvlášť.")

Levý graf je fázový přechod, jak se rodí: pro malé mřížky plynulá křivka, pro velké skoro schod. V limitě $L \to \infty$ by to byl skutečný skok z nuly na jedničku v bodě $p_c = 1/2$.

Pravý graf je přesná kontrola — a je na ní poučné, jak snadno se dá minout. Samodualita opravdu dává **přesně $1/2$**, ale platí pro obdélník o rozměrech $L \times (L+1)$, tedy o jeden sloupec širší, než je vysoký. Na čtverci je vodorovný směr nepatrně zvýhodněný, takže vychází víc než $1/2$ a k jedné polovině klesá až s rostoucím $L$.

Zelená řada v grafu stojí na $1/2$ pro každé $L$ zvlášť, modrá se k ní teprve blíží. Kdybychom měřili jen na čtverci, mohli bychom z těch 0.52 usoudit, že $p_c$ leží malinko jinde.

## 3. Jak velký je největší shluk

Druhý pohled na tutéž věc: místo „proteče?" se ptejme, jaká část mřížky patří do **největší propojené komponenty**. Pod $p_c$ jsou všechny shluky malé, nad $p_c$ existuje jeden obří, který pohltí konečný podíl mřížky.

In [ ]:
def nejvetsi_shluk(L, p):
    """Podíl uzlů v největší souvislé komponentě."""
    H, V = mrizka(L, L, p)
    videno = [[False] * L for _ in range(L)]
    nejvetsi = 0
    for r0 in range(L):
        for c0 in range(L):
            if videno[r0][c0]:
                continue
            velikost, zasobnik = 0, [(r0, c0)]
            videno[r0][c0] = True
            while zasobnik:
                r, c = zasobnik.pop()
                velikost += 1
                for rr, cc, otevreno in ((r, c + 1, c < L - 1 and H[r][c]),
                                         (r, c - 1, c > 0 and H[r][c - 1]),
                                         (r + 1, c, r < L - 1 and V[r][c]),
                                         (r - 1, c, r > 0 and V[r - 1][c])):
                    if otevreno and not videno[rr][cc]:
                        videno[rr][cc] = True
                        zasobnik.append((rr, cc))
            nejvetsi = max(nejvetsi, velikost)
    return nejvetsi / (L * L)


fig, ax = plt.subplots(figsize=(7.5, 4.3))
for L, barva in ((16, "tab:blue"), (32, "tab:orange"), (64, "tab:green")):
    podily = [sum(nejvetsi_shluk(L, p) for _ in range(30 if L < 64 else 12)) / (30 if L < 64 else 12)
              for p in pecka]
    ax.plot(pecka, podily, "o-", ms=3, color=barva, label=f"$L$ = {L}")
ax.axvline(0.5, color="k", ls=":", lw=1)
ax.set_xlabel("$p$")
ax.set_ylabel("podíl mřížky v největším shluku")
ax.set_title("Pod $p_c$ samé ostrůvky, nad ním jeden obří shluk", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Perkolace | každá vazba otevřená s pravděpodobností $p$, nezávisle |
| Fázový přechod | pod $p_c$ průsak prakticky nemožný, nad ním prakticky jistý |
| $p_c = 1/2$ | pro vazbovou perkolaci na čtvercové mřížce **přesně**, ze samoduality |
| Přesná kontrola | při $p = 1/2$ je P(průsak) = 1/2 pro každé $L$ — ale jen na obdélníku $L\times(L+1)$ |
| Konečná mřížka | přechod je rozmazaný; ostří se s rostoucím $L$ |
| Největší shluk | pod $p_c$ jen ostrůvky, nad ním jeden obří |
| Kde se to hodí | šíření požáru, poruchy v síti, průsak horninou, odolnost materiálů |

Souvislost s ostatními sešity: je to týž fázový přechod jako u větvícího procesu, jen v prostoru. Tam rozhodoval střední počet potomků $R_0$, tady hustota otevřených vazeb $p$ — a v obou případech se chování v jediném bodě zlomí.

### Zkuste si

1. Udělejte **uzlovou** perkolaci (otevírají se uzly, ne vazby). Kritická hodnota už není 1/2 — změřte ji. (Vyjde kolem 0.5927, a uzavřený vzorec se pro ni nezná.)
2. Zkuste trojúhelníkovou mřížku, kde uzlová perkolace má zase přesně $p_c = 1/2$.
3. Změřte, jak rychle se přechod zostřuje: šířka přechodové oblasti klesá jako $L^{-1/\nu}$. Odhadněte $\nu$ (pravá hodnota je 4/3).
4. Ve třech rozměrech je $p_c \approx 0.2488$. Ověřte, že v prostoru stačí k průsaku mnohem méně otevřených vazeb, a vysvětlete proč.
5. Místo „proteče?" sledujte **délku nejkratší cesty** napříč. Jak se chová blízko $p_c$?
6. Model požáru: zapalte levý okraj a sledujte, jak se oheň šíří v čase. Kolik kroků trvá průchod v závislosti na $p$?